In [7]:
import os
import sys

import pandas as pd
import plotly.express as px

sys.path.insert(0, 'notebook_styling')
import psrc_theme  # noqa: F401 — registers PSRC plotly template and renderer

from util import CensusApi, compute_headship_rates, YEAR_CONFIG

c = CensusApi(os.getenv("CENSUS_KEY"), timeout=90)


In [8]:
metro_ids = [42660, 41860, 31080, 41940, 40140, 41740, 38900, 19740, 38060, 29820]

MSA_COL = 'metropolitan statistical area/micropolitan statistical area'


def fetch_metro_totals(cols_dict, year, dataset, msa_ids=None):
    """Pull a decennial table for metropolitan statistical areas and return a
    DataFrame indexed by CBSA code with one column per age group.

    Parameters
    ----------
    cols_dict : dict  mapping age-group labels to lists of Census variable codes
    year      : int   decennial year (2000, 2010, 2020)
    dataset   : str   Census dataset slug (e.g. 'sf1', 'sf4', 'dhc')
    msa_ids   : list  CBSA codes as ints or strings (defaults to module-level metro_ids)
    """
    if msa_ids is None:
        msa_ids = metro_ids
    all_vars = ['GEO_ID', 'NAME'] + [v for vals in cols_dict.values() for v in vals]
    for_predicates = f'{MSA_COL}:{",".join(str(m) for m in msa_ids)}'
    df = c.get_table(all_vars, year, for_predicates, None, f'dec/{dataset}')
    df = CensusApi.combine_groups(cols_dict, df)
    df['geoid'] = df['GEO_ID'].str.slice(start=-5).astype(int)
    df = df.rename(columns={'NAME': 'name'}).set_index('geoid')
    return df[['name'] + list(cols_dict.keys())]

In [9]:
metro_headship_rates = compute_headship_rates(
    {k: YEAR_CONFIG[k] for k in (2010, 2020)},
    fetch_fn=fetch_metro_totals,
    geo_label='metro_id',
)

In [10]:
# Fetch metro names and derive an abbreviated label (first city in the MSA name)
_names = c.get_table(
    ['GEO_ID', 'NAME'],
    2020,
    f'{MSA_COL}:{",".join(str(m) for m in metro_ids)}',
    None,
    'dec/dhc',
)
_names['metro_id'] = _names['GEO_ID'].str.slice(start=-5).astype(int)
_names['metro_name'] = _names['NAME'].str.split('-').str[0].str.split(',').str[0].str.strip()
_names = _names.set_index('metro_id')['metro_name']

out = metro_headship_rates.reset_index()
out.insert(1, 'metro_name', out['metro_id'].map(_names))
out.to_csv('data/metro_headship_rates_by_age.csv', index=False)

# Per Metro

In [11]:
plot_df = out.melt(
    id_vars=['metro_id', 'metro_name', 'age'],
    value_vars=['headship_rate_2010', 'headship_rate_2020'],
    var_name='year',
    value_name='headship_rate',
)
plot_df['year'] = plot_df['year'].str.extract(r'(\d{4})').astype(int)

metro_name_order = sorted(plot_df['metro_name'].unique())

for age, group in plot_df.groupby('age'):
    fig = px.line(
        group.sort_values('year'),
        x='year',
        y='headship_rate',
        color='metro_name',
        markers=True,
        title=age,
        labels={'headship_rate': 'Headship Rate', 'year': 'Year', 'metro_name': 'Metro'},
        category_orders={'metro_name': metro_name_order},
    )
    fig.update_xaxes(tickvals=[2010, 2020], tickformat='d')
    fig.update_traces(line=dict(width=1), opacity=0.4)
    fig.update_traces(
        selector=dict(name='Seattle'),
        line=dict(width=4),
        opacity=1.0,
    )
    fig.show()


# Seattle vs. Peers

In [12]:
seattle_id = 42660

# Mean across peer metros (excluding Seattle) per age/year
metro_mean = (
    plot_df[plot_df['metro_id'] != seattle_id]
    .groupby(['age', 'year'])['headship_rate']
    .mean()
    .reset_index()
    .assign(series='Peer Metro Mean')
)
seattle = (
    plot_df[plot_df['metro_id'] == seattle_id][['age', 'year', 'headship_rate']]
    .assign(series='Seattle')
)
compare_df = pd.concat([seattle, metro_mean], ignore_index=True)

for age, group in compare_df.groupby('age'):
    fig = px.line(
        group.sort_values('year'),
        x='year',
        y='headship_rate',
        color='series',
        markers=True,
        title=age,
        labels={'headship_rate': 'Headship Rate', 'year': 'Year', 'series': ''},
    )
    fig.update_xaxes(tickvals=[2010, 2020], tickformat='d')
    fig.show()
